### Kellerautomaten

[Video](https://youtu.be/-aVF00KVMns)

Dieses Notebook setzt die Notebooks **Reguläre Sprachen** und **Grammatiken** voraus.

Endliche Automaten stoßen an ihre Grenzen, wenn sie sich beliebig viel merken müssten. Ein Beispiel ist die Sprache
$L = \{0^n1^n \mid n \ge 0\}$. Hier muss der Automat wissen, wie viele Nullen er gesehen hat, um später genau gleich
viele Einsen zu prüfen. Das kann ein endlicher Automat nicht, ihm fehlt das Gedächtnis.

Ein **Kellerautomat** ist wie ein endlicher Automat, aber mit einem zusätzlichen Speicher, dem **Keller** (Stapel,
Stack, siehe **ADT Keller und Schlange**). Er kann oben etwas auf den Keller legen (push) und das oberste Element
entfernen (pop). Es ist immer nur das oberste Element sichtbar.

Ein Kellerautomat kann in einem Schritt

- ein Eingabezeichen lesen (oder auch keins),
- den Zustand wechseln,
- das oberste Kellerzeichen durch eine Zeichenfolge ersetzen und damit etwas ablegen oder entfernen.

Für $L = \{0^n1^n\}$: Für jede gelesene $0$ lege ein Zeichen in den Keller, für jede gelesene $1$ entferne eins. Am
Ende muss der Keller wieder leer sein.

#### Definition

Ein **nichtdeterministischer Kellerautomat** (NPDA, *nondeterministic pushdown automaton*) besteht aus

1. $Q$, einer endlichen Menge von Zuständen,
2. $\Sigma$, einem endlichen Eingabealphabet,
3. $\Gamma$, einem endlichen Kelleralphabet,
4. $\delta$, einer Menge von Übergängen,
5. $s \in Q$, einem Startzustand,
6. $\# \in \Gamma$, dem Kellerboden: Am Anfang liegt nur $\#$ im Keller,
7. $F \subseteq Q$, einer Menge von Endzuständen.

Ein Übergang hat die Form $(q, a, X) \rightarrow (q', \gamma)$. Das liest man so: Wenn ich im Zustand $q$ bin, das
Zeichen $a$ lese und oben im Keller $X$ liegt, dann gehe ich in den Zustand $q'$ und ersetze $X$ durch die
Zeichenfolge $\gamma$ aus $\Gamma^*$.

Den Keller schreiben wir als Zeichenfolge, das oberste Zeichen steht links. $0\#$ heißt: oben liegt $0$, darunter der
Boden.

- $(q, 0, \#) \rightarrow (q, 0\#)$: lese $0$ und lege $0$ auf den Keller
- $(q, 0, 0) \rightarrow (q, 00)$: lese $0$ und lege eine weitere $0$ auf den Keller
- $(q, 1, 0) \rightarrow (q, \varepsilon)$: lese $1$ und entferne eine $0$ vom Keller
- $(q, \varepsilon, X) \rightarrow (q', X)$: wechsle den Zustand, ohne etwas zu lesen, der Keller bleibt gleich

Das oberste Zeichen wird also immer weggenommen. Soll es liegen bleiben, schreibt man es in $\gamma$ wieder hin.
Die Zeichen im Keller können andere sein als die der Eingabe. Oft nimmt man für die Eingabe Kleinbuchstaben und für
den Keller Großbuchstaben.

#### Zustandsdiagramm

Im Diagramm schreiben wir an die Pfeile $a, X \rightarrow \gamma$: Zeichen lesen, oberstes Kellerzeichen, Ersatz.

<img src="keller_beispiel.png" width="560">

Ein Kellerautomat **akzeptiert** ein Wort, wenn es eine Möglichkeit gibt, das ganze Wort zu lesen und dabei in einen
Endzustand zu kommen. Gibt es für eine Situation keinen passenden Übergang, bleibt der Automat stecken, dieser Weg
akzeptiert nicht.

Die Abarbeitung beschreibt man mit **Konfigurationen**: Zustand, noch zu lesendes Restwort und Kellerinhalt.

| Zustand | Restwort | Keller |
|---|---|---|
| $q_0$ | $aabbb$ | $\#$ |
| $q_1$ | $abbb$ | $a\#$ |
| $q_1$ | $bbb$ | $aa\#$ |
| $q_2$ | $bb$ | $a\#$ |
| $q_2$ | $b$ | $\#$ |
| $q_3$ | $\varepsilon$ | $\#$ |

Das Wort $aabbb$ wird akzeptiert. Allgemein muss das Wort mit mindestens einem $a$ beginnen, dann kommen nur noch
$b$. Jedes $b$ nimmt ein $a$ vom Keller, und nach $q_3$ kommt man nur mit einem weiteren $b$, wenn der Keller schon
leer ist. Der Automat akzeptiert also

$$L = \{a^nb^m \mid n \ge 1,\ m > n\}$$

#### JFLAP

Mit dem Programm [JFLAP](https://www.jflap.org) kann man Kellerautomaten zeichnen und testen. Dort wird der Pfeil
durch einen Strichpunkt ersetzt, $\varepsilon$ heißt $\lambda$ und der Kellerboden heißt $Z$. Oben JFLAP, unten
dasselbe Beispiel von Hand:

<img src="bild13.png" width="602">

Die Datei `kellerautomat1.jff` enthält dieses Beispiel für JFLAP.

### Kellerautomaten in Python

Ein Übergang ist ein Tupel `(q, a, X, q2, gamma)`, der leere String steht für $\varepsilon$. Den Keller speichern wir
als String, oben ist links.

In [1]:
beispiel = {
    'start': 'q0',
    'ende': {'q3'},
    'uebergaenge': [
        ('q0', 'a', '#', 'q1', 'a#'),
        ('q1', 'a', 'a', 'q1', 'aa'),
        ('q1', 'b', 'a', 'q2', ''),
        ('q2', 'b', 'a', 'q2', ''),
        ('q2', 'b', '#', 'q3', '#'),
        ('q3', 'b', '#', 'q3', '#'),
    ],
}

def nachfolger(k, konf):
    '''alle Konfigurationen, die in einem Schritt erreichbar sind'''
    zustand, rest, keller = konf
    ergebnis = []
    for q, a, x, q2, gamma in k['uebergaenge']:
        if q == zustand and rest.startswith(a) and keller.startswith(x):
            ergebnis.append((q2, rest[len(a):], gamma + keller[1:]))
    return ergebnis

nachfolger(beispiel, ('q1', 'abbb', 'a#'))

[('q1', 'bbb', 'aa#')]

Weil ein Kellerautomat nichtdeterministisch sein darf, kann es mehrere Nachfolger geben. Wir probieren alle Wege mit
einer Breitensuche aus, wie bei der Suche nach einer Ableitung im Notebook **Grammatiken**. Damit die Suche bei
$\varepsilon$-Übergängen nicht endlos läuft, begrenzen wir die Höhe des Kellers.

In [2]:
def ablauf(k, wort):
    '''eine akzeptierende Folge von Konfigurationen, None wenn das Wort nicht akzeptiert wird'''
    start = (k['start'], wort, '#')
    vorher = {start: None}
    offen = [start]
    while offen:
        naechste = []
        for konf in offen:
            zustand, rest, keller = konf
            if rest == '' and zustand in k['ende']:
                weg = []
                while konf is not None:
                    weg.append(konf)
                    konf = vorher[konf]
                return weg[::-1]
            for neu in nachfolger(k, konf):
                if neu not in vorher and len(neu[2]) <= 2 * len(wort) + 4:
                    vorher[neu] = konf
                    naechste.append(neu)
        offen = naechste
    return None

def zeige(k, wort):
    weg = ablauf(k, wort)
    if weg is None:
        print(wort, 'wird nicht akzeptiert')
        return
    for zustand, rest, keller in weg:
        print(f'{zustand:4}{rest or "ε":10}{keller}')

zeige(beispiel, 'aabbb')
zeige(beispiel, 'aabb')

q0  aabbb     #
q1  abbb      a#
q1  bbb       aa#
q2  bb        a#
q2  b         #
q3  ε         #
aabb wird nicht akzeptiert


Damit können wir prüfen, ob der Automat genau die Wörter $a^nb^m$ mit $n \ge 1$ und $m > n$ akzeptiert:

In [3]:
import itertools
import re

def alle_woerter(alphabet, n):
    for laenge in range(n + 1):
        for t in itertools.product(alphabet, repeat=laenge):
            yield ''.join(t)

def in_L(w):
    m = re.fullmatch('(a+)(b+)', w)
    return m is not None and len(m[2]) > len(m[1])

fehler = [w for w in alle_woerter('ab', 10) if (ablauf(beispiel, w) is not None) != in_L(w)]
print(len(fehler), 'Fehler')
print([w for w in alle_woerter('ab', 6) if ablauf(beispiel, w)])

0 Fehler
['abb', 'abbb', 'aabbb', 'abbbb', 'aabbbb', 'abbbbb']


### Weitere Beispiele

#### Gleich viele $a$ und $b$

Die Wörter über $\{a, b\}$ mit gleich vielen $a$ und $b$, in beliebiger Reihenfolge. Der Keller merkt sich den
Überschuss: Liegen $A$ im Keller, hat der Automat bisher mehr $a$ gelesen, liegen $B$ darin, mehr $b$. Ein Zeichen,
das zum Überschuss passt, vergrößert ihn, das andere hebt eins auf. Ist der Keller leer, darf er in den Endzustand.

<img src="keller_gleich.png" width="330">

In [4]:
gleich = {
    'start': 'q0',
    'ende': {'q1'},
    'uebergaenge': [
        ('q0', 'a', '#', 'q0', 'A#'), ('q0', 'a', 'A', 'q0', 'AA'), ('q0', 'a', 'B', 'q0', ''),
        ('q0', 'b', '#', 'q0', 'B#'), ('q0', 'b', 'B', 'q0', 'BB'), ('q0', 'b', 'A', 'q0', ''),
        ('q0', '', '#', 'q1', '#'),
    ],
}
zeige(gleich, 'abbbaa')
fehler = [w for w in alle_woerter('ab', 10) if (ablauf(gleich, w) is not None) != (w.count('a') == w.count('b'))]
print(len(fehler), 'Fehler')

q0  abbbaa    #
q0  bbbaa     A#
q0  bbaa      #
q0  baa       B#
q0  aa        BB#
q0  a         B#
q0  ε         #
q1  ε         #
0 Fehler


#### Palindrome: der Automat muss raten

Ein Palindrom gerader Länge hat die Form $ww^R$, wobei $w^R$ das umgedrehte Wort $w$ ist, z.B. $abba$ oder $baab$.
Der Automat legt die erste Hälfte auf den Keller und vergleicht danach die zweite Hälfte mit dem Keller. Der Keller
gibt die Zeichen in umgekehrter Reihenfolge zurück, genau passend.

Woher weiß der Automat, wo die Mitte ist? Er weiß es nicht. Mit einem $\varepsilon$-Übergang darf er an jeder Stelle
in den Vergleichszustand $q_1$ wechseln. Das Wort wird akzeptiert, wenn eine der Möglichkeiten zum Ziel führt.

<img src="keller_palindrom.png" width="450">

In [5]:
palindrom = {
    'start': 'q0',
    'ende': {'q2'},
    'uebergaenge': [('q0', a, x, 'q0', a + x) for x in '#ab' for a in 'ab']
                   + [('q0', '', x, 'q1', x) for x in '#ab']
                   + [('q1', 'a', 'a', 'q1', ''), ('q1', 'b', 'b', 'q1', ''), ('q1', '', '#', 'q2', '#')],
}
zeige(palindrom, 'abbbba')
zeige(palindrom, 'abab')
fehler = [w for w in alle_woerter('ab', 10)
          if (ablauf(palindrom, w) is not None) != (w == w[::-1] and len(w) % 2 == 0)]
print(len(fehler), 'Fehler')

q0  abbbba    #
q0  bbbba     a#
q0  bbba      ba#
q0  bba       bba#
q1  bba       bba#
q1  ba        ba#
q1  a         a#
q1  ε         #
q2  ε         #
abab wird nicht akzeptiert
0 Fehler


### Deterministisch und nichtdeterministisch

Ein Kellerautomat ist **deterministisch** (DPDA), wenn es in jeder Situation höchstens einen möglichen Übergang gibt.
Dazu zählt auch: Gibt es für einen Zustand und ein oberstes Kellerzeichen einen $\varepsilon$-Übergang, dann darf es
dafür keinen Übergang geben, der ein Zeichen liest.

Das erste Beispiel ist deterministisch, die beiden anderen nicht: Bei *gleich viele* kann der Automat mit leerem
Keller weiterlesen oder in den Endzustand gehen, bei den Palindromen muss er die Mitte raten.

In [6]:
def deterministisch(k):
    for u, v in itertools.combinations(k['uebergaenge'], 2):
        gleiche_lage = u[0] == v[0] and u[2] == v[2]
        if gleiche_lage and (u[1] == v[1] or u[1] == '' or v[1] == ''):
            return False
    return True

for name, k in [('beispiel', beispiel), ('gleich', gleich), ('palindrom', palindrom)]:
    print(name, deterministisch(k))

beispiel True
gleich False
palindrom False


Bei endlichen Automaten macht das keinen Unterschied, zu jedem NEA gibt es einen DEA (Potenzmengenkonstruktion).
Bei Kellerautomaten schon:

- Für die Wörter mit gleich vielen $a$ und $b$ gibt es auch einen deterministischen Kellerautomaten. Er braucht
  einen zweiten Zustand: In den Endzustand geht er mit $\varepsilon$, sobald der Keller leer ist, und liest erst von
  dort aus weiter.
- Steht in der Mitte ein Trennzeichen, also $w\,c\,w^R$, dann muss der Automat nicht raten. Dafür gibt es einen
  deterministischen Kellerautomaten.
- Für die Palindrome $ww^R$ ohne Trennzeichen gibt es keinen deterministischen Kellerautomaten. Das zu beweisen ist
  aufwendig.

Deterministische Kellerautomaten erkennen also weniger Sprachen als nichtdeterministische.

### Von der Grammatik zum Kellerautomaten

Nichtdeterministische Kellerautomaten erkennen genau die **kontextfreien** Sprachen. Eine Richtung kann man leicht
einsehen: Zu jeder kontextfreien Grammatik gibt es einen Kellerautomaten mit drei Zuständen, der eine Linksableitung
im Keller nachspielt.

- $q_0 \rightarrow q_1$: $\varepsilon, \# \rightarrow S\#$, das Startsymbol kommt in den Keller.
- In $q_1$ für jede Regel $A \rightarrow \alpha$: $\varepsilon, A \rightarrow \alpha$. Liegt oben ein Nichtterminal,
  wird es durch eine rechte Seite ersetzt (geraten).
- In $q_1$ für jedes Terminal $a$: $a, a \rightarrow \varepsilon$. Liegt oben ein Terminal, muss es das nächste
  Zeichen der Eingabe sein.
- $q_1 \rightarrow q_2$: $\varepsilon, \# \rightarrow \#$. Ist der Keller leer, ist die Ableitung fertig.

Für $S \rightarrow 0S1 \mid \varepsilon$:

<img src="keller_grammatik.png" width="420">

In [7]:
def grammatik2keller(regeln):
    terminale = sorted({z for seiten in regeln.values() for s in seiten for z in s if not z.isupper()})
    uebergaenge = [('q0', '', '#', 'q1', 'S#')]
    for links, seiten in regeln.items():
        for rechts in seiten:
            uebergaenge.append(('q1', '', links, 'q1', rechts))
    for a in terminale:
        uebergaenge.append(('q1', a, a, 'q1', ''))
    uebergaenge.append(('q1', '', '#', 'q2', '#'))
    return {'start': 'q0', 'ende': {'q2'}, 'uebergaenge': uebergaenge}

k = grammatik2keller({'S': ['0S1', '']})
zeige(k, '0011')

q0  0011      #
q1  0011      S#
q1  0011      0S1#
q1  011       S1#
q1  011       0S11#
q1  11        S11#
q1  11        11#
q1  1         1#
q1  ε         #
q2  ε         #


Im Keller steht nach jedem $\varepsilon$-Schritt der noch nicht gelesene Teil der Satzform: $S$, $0S1$, $S1$, $0S11$,
... Zusammen mit dem schon gelesenen Teil ergibt sich die Linksableitung
$S \Rightarrow 0S1 \Rightarrow 00S11 \Rightarrow 0011$.

Dasselbe mit den korrekt geklammerten Ausdrücken aus dem Notebook **Grammatiken**:

In [8]:
k = grammatik2keller({'S': ['(S)S', '']})
print([w for w in alle_woerter('()', 6) if ablauf(k, w)])

['', '()', '(())', '()()', '((()))', '(()())', '(())()', '()(())', '()()()']


Umgekehrt kann man aus jedem Kellerautomaten eine kontextfreie Grammatik konstruieren, das ist aber aufwendiger.
Insgesamt gilt:

> Eine Sprache ist genau dann kontextfrei, wenn ein nichtdeterministischer Kellerautomat sie erkennt.

Ein Kellerautomat kann statt mit Endzuständen auch mit leerem Keller akzeptieren (das Wort ist gelesen und der Keller
ganz leer). Beide Arten erkennen dieselben Sprachen.

### Grenzen kontextfreier Sprachen

Ein Kellerautomat hat nur einen Keller, und an die unteren Zeichen kommt er erst, wenn er die oberen entfernt hat.
Damit kann er eine Anzahl mit *einer* anderen vergleichen, aber nicht mit zwei.

- $\{a^nb^nc^n \mid n \ge 1\}$ ist nicht kontextfrei: Hat der Automat die $a$ gezählt und beim Lesen der $b$ wieder
  abgebaut, weiß er nicht mehr, wie viele $c$ kommen müssen. Eine kontextsensitive Grammatik dafür steht im
  Notebook **Grammatiken**.
- $\{ww \mid w \in \{a, b\}^*\}$ ist nicht kontextfrei, obwohl $\{ww^R\}$ es ist: Der Keller liefert die Zeichen in
  umgekehrter Reihenfolge, für $ww$ bräuchte man sie in derselben Reihenfolge.

Beweisen kann man das mit einem Pumping-Lemma für kontextfreie Sprachen, ähnlich wie bei den regulären Sprachen.

#### Abschlusseigenschaften

Sind $L_1$ und $L_2$ kontextfrei mit den Startsymbolen $S_1$ und $S_2$, dann auch

- $L_1 \cup L_2$: $S \rightarrow S_1 \mid S_2$
- $L_1L_2$: $S \rightarrow S_1S_2$
- $L_1^*$: $S \rightarrow S_1S \mid \varepsilon$

Der Durchschnitt zweier kontextfreier Sprachen ist aber nicht immer kontextfrei:

$L_1 = \{a^nb^nc^m \mid n, m \ge 0\}$ und $L_2 = \{a^mb^nc^n \mid n, m \ge 0\}$ sind kontextfrei, ihr Durchschnitt ist
$\{a^nb^nc^n \mid n \ge 0\}$.

In [9]:
def woerter(regeln, n):
    ergebnis, offen, gesehen = set(), ['S'], {'S'}
    while offen:
        form = offen.pop()
        i = next((k for k, z in enumerate(form) if z.isupper()), None)
        if i is None:
            ergebnis.add(form)
            continue
        for rechts in regeln[form[i]]:
            neu = form[:i] + rechts + form[i + 1:]
            if sum(1 for z in neu if not z.isupper()) <= n and len(neu) <= 2 * n + 4 and neu not in gesehen:
                gesehen.add(neu)
                offen.append(neu)
    return ergebnis

L1 = {'S': ['AC'], 'A': ['aAb', ''], 'C': ['cC', '']}
L2 = {'S': ['AB'], 'A': ['aA', ''], 'B': ['bBc', '']}
sorted(woerter(L1, 9) & woerter(L2, 9), key=len)

['', 'abc', 'aabbcc', 'aaabbbccc']

Auch das Komplement einer kontextfreien Sprache ist nicht immer kontextfrei. Sonst wäre wegen
$L_1 \cap L_2 = \overline{\overline{L_1} \cup \overline{L_2}}$ auch jeder Durchschnitt kontextfrei. Reguläre Sprachen
sind dagegen unter allen diesen Operationen abgeschlossen.

### Mehr Speicher

| Modell | Speicher | Sprachen |
|---|---|---|
| endlicher Automat | nur der Zustand | regulär (Typ 3) |
| Kellerautomat | ein Keller | kontextfrei (Typ 2) |
| linear beschränkter Automat | Band so lang wie die Eingabe | kontextsensitiv (Typ 1) |
| Turingmaschine | unbegrenztes Band | Typ 0 |

Ein Kellerautomat mit **zwei** Kellern ist schon so mächtig wie eine Turingmaschine: Keller 1 speichert alles links
vom Lesekopf, Keller 2 alles rechts davon. Bewegt sich der Kopf nach rechts, wandert ein Zeichen von Keller 2 auf
Keller 1, und umgekehrt. Turingmaschinen erkennen genau die Sprachen vom Typ 0.

### Anwendungen

- **Klammern prüfen:** Editoren und Compiler prüfen mit einem Keller, ob Klammern wie `(`, `[`, `{` richtig
  geschlossen werden.
- **Parser:** Die Syntax von Programmiersprachen ist (weitgehend) kontextfrei. Parser arbeiten deshalb mit einem
  Keller. Für Programmiersprachen wählt man Grammatiken, die ein *deterministischer* Kellerautomat verarbeiten kann,
  damit nicht geraten werden muss.
- **Rekursion:** Der Parser mit rekursivem Abstieg aus dem Notebook **Grammatiken** hat keinen eigenen Keller. Er
  benutzt den Aufrufstapel von Python: Jeder Aufruf von `ausdruck`, `term` oder `faktor` liegt dort, bis er fertig
  ist.

### Zusammenfassung

- Ein **Kellerautomat** ist ein endlicher Automat mit einem Keller. Ein Übergang $a, X \rightarrow \gamma$ liest $a$
  (oder nichts), nimmt das oberste Kellerzeichen $X$ weg und legt $\gamma$ ab.
- Er **akzeptiert** ein Wort, wenn es einen Weg gibt, der das ganze Wort liest und in einem Endzustand endet. Die
  Abarbeitung beschreibt man mit Konfigurationen (Zustand, Restwort, Keller).
- **Nichtdeterministische** Kellerautomaten erkennen genau die **kontextfreien** Sprachen. Deterministische erkennen
  weniger, z.B. nicht die Palindrome $ww^R$.
- Zu einer kontextfreien Grammatik bekommt man einen Kellerautomaten, der eine Linksableitung im Keller nachspielt.
- **Grenzen:** $\{a^nb^nc^n\}$ und $\{ww\}$ sind nicht kontextfrei. Der Durchschnitt zweier kontextfreier Sprachen
  ist nicht immer kontextfrei.